In [ ]:
# add reload option:
%reload_ext autoreload
%autoreload 2

# sample points and then place the circle on it. 
import geolipi.symbolic as gls
import numpy as np
import matplotlib.pyplot as plt
import torch as th
from migumi.torch_compute.polyline_utils import get_expr_bounds
from geolipi.torch_compute.sketcher import Sketcher
from geolipi.torch_compute import recursive_evaluate

# mill path
import sys
sys.path.append("/users/aganesh8/data/aganesh8/projects/kigumi/decor_gumi")
from polyarc_csg.offset import get_offset_expr
from decor_gumi.polyarc_sampler import sample_polyarc
from migumi.utils.vis import draw_contour_plot, draw_solid_overlay, fig_to_image, overlay_image, frames_to_animation


res = 1024
sketcher = Sketcher(n_dims=2, device='cuda:0', resolution=res)
DEFAULT_MILLBIT_SIZE = 0.21166666666666667 # Quarter inch millbit size in our units.

part_colors = [
    (0.267, 0.444, 0.672, 1.00),
    (0.33, 0.8, 0.27, 1.00),
    (0.710, 0.388, 0.336, 1.00),
    (0.498, 0.592, 1.00, 1.00),
    (0.710, 0.608, 0.569, 1.00),
    (0.769, 0.306, 0.863, 1.00),
    (1.00, 0.298, 0.420, 1.00),
]
subtr_color = (0.774, 0.831, 0.133, 1.0)
drillbit_color = "#dddddd"
part_color_1 = part_colors[6]
part_color_2 = part_colors[3]

In [ ]:
active_part_timber = {
    "main_1": gls.PolyArc2D(th.tensor([[ 2.2990e-08,  5.0000e-01,  0.0000e+00],
        [ 4.9097e-07, -5.0000e-01,  0.0000e+00],
        [ 2.0000e+00, -5.0000e-01,  0.0000e+00],
        [ 2.0000e+00,  5.0000e-01,  0.0000e+00],
        [ 2.2990e-08,  5.0000e-01,  0.0000e+00]], device='cuda:0')),
    "main_2": gls.PolyArc2D(th.tensor([[ 1.0000,  0.5000,  0.0000],
        [ 1.0000, -0.5000,  0.0000],
        [ 3.0000, -0.5000,  0.0000],
        [ 3.0000,  0.5000,  0.0000],
        [ 1.0000,  0.5000,  0.0000]], device='cuda:0'))
}

subtr_1_arg = th.tensor([[ 1.2000,  0.8000,  0.0000],
        [ 2.2000,  0.8000,  0.0000],
        [ 2.2000, -0.8000,  0.0000],
        [ 1.2000, -0.8000,  0.0000],
        [ 1.2000, -0.1671,  0.0000],
        [ 1.8020, -0.3000,  0.0000],
        [ 1.8020,  0.3000,  0.0000],
        [ 1.2000,  0.1671,  0.0000]], device='cuda:0')
subtr_1_arg.roll(2, dims=0)         
subtr = {
    "main_1": gls.PolyArc2D(subtr_1_arg),
    "main_2": gls.PolyArc2D(th.tensor([[ 1.2000,  0.1671,  0.0000],
        [ 1.2000,  0.8000,  0.0000],
        [ 0.8000,  0.8000,  0.0000],
        [ 0.8000, -0.8000,  0.0000],
        [ 1.2000, -0.8000,  0.0000],
        [ 1.2000, -0.1671,  0.0000],
        [ 1.8020, -0.3000,  0.0000],
        [ 1.8020,  0.3000,  0.0000]], device='cuda:0'))
}

subtr_odf = {
    "main_1": gls.PolyArc2D(th.tensor([[ 1.8020,  0.1703,  0.0000],
        [ 1.8020, -0.1703, -0.4675],
        [ 1.6754, -0.2720,  0.0000],
        [ 1.3287, -0.1955,  0.4794],
        [ 1.2000, -0.2988,  0.0000],
        [ 1.2000, -0.6942,  0.4142],
        [ 1.3058, -0.8000,  0.0000],
        [ 2.0942, -0.8000,  0.4142],
        [ 2.2000, -0.6942,  0.0000],
        [ 2.2000,  0.6942,  0.4142],
        [ 2.0942,  0.8000,  0.0000],
        [ 1.3058,  0.8000,  0.4142],
        [ 1.2000,  0.6942,  0.0000],
        [ 1.2000,  0.2988,  0.4794],
        [ 1.3287,  0.1955,  0.0000],
        [ 1.6754,  0.2720, -0.4675]], device='cuda:0')),
    "main_2": gls.PolyArc2D(th.tensor([[ 1.2000,  0.2968,  0.0000],
        [ 1.2000,  0.6942,  0.4142],
        [ 1.0942,  0.8000,  0.0000],
        [ 0.9058,  0.8000,  0.4142],
        [ 0.8000,  0.6942,  0.0000],
        [ 0.8000, -0.6942,  0.4142],
        [ 0.9058, -0.8000,  0.0000],
        [ 1.0942, -0.8000,  0.4142],
        [ 1.2000, -0.6942,  0.0000],
        [ 1.2000, -0.2968, -0.4675],
        [ 1.3266, -0.1951,  0.0000],
        [ 1.6733, -0.2716,  0.4794],
        [ 1.8020, -0.1683,  0.0000],
        [ 1.8020,  0.1683,  0.4794],
        [ 1.6733,  0.2716,  0.0000],
        [ 1.3266,  0.1951, -0.4675]], device='cuda:0'))
}
part_1 = gls.Difference(active_part_timber["main_1"],subtr_odf["main_1"])
part_2 = gls.Difference(active_part_timber["main_2"],subtr_odf["main_2"])


inset_expr = get_offset_expr(subtr["main_1"], DEFAULT_MILLBIT_SIZE/2.0)
opening_expr = get_offset_expr(inset_expr, - DEFAULT_MILLBIT_SIZE/2.0 - 1e-5)

mill_part_1 = gls.Difference(active_part_timber["main_1"], opening_expr)

inset_expr = get_offset_expr(subtr["main_2"], DEFAULT_MILLBIT_SIZE/2.0)
opening_expr = get_offset_expr(inset_expr, - DEFAULT_MILLBIT_SIZE/2.0 - 1e-5)

mill_part_2 = gls.Difference(active_part_timber["main_2"], opening_expr)

In [ ]:

design = subtr["main_1"]
inset_expr = get_offset_expr(design, DEFAULT_MILLBIT_SIZE/2.0)
# opening_expr = get_offset_expr(inset_expr, - DEFAULT_MILLBIT_SIZE/2.0 - 1e-5)

sample_points, _, _ = sample_polyarc(inset_expr, density=20,)
sample_points.shape

In [ ]:

scale, origin = get_expr_bounds(gls.Union(part_1, part_2), sketcher)
scale = [x * 2 for x in scale]
coords, shape = sketcher.create_non_square_coords(scale, origin)
coords = coords.float()
figsize = (scale[1] * 3, scale[0] * 3)

# N = sample_points.shape[0]
# seq = []
# for i in range(N):
#     cur_translation = sample_points[i]
#     drill_bit = gls.Circle2D((DEFAULT_MILLBIT_SIZE/2.0,))
#     drill_bit = gls.Translate2D(drill_bit, cur_translation)
#     output2 = recursive_evaluate(drill_bit.tensor(), sketcher, coords=coords)
#     output2 = output2.cpu().numpy()
#     output2 = output2.reshape(shape)[..., None]
#     fig = draw_solid_overlay(output2, sketcher.resolution, figsize=figsize, color=drillbit_color)
#     img2 = fig_to_image(fig)
#     seq.append(img2)
# output_path = "renders/vis_1_drill_1.webp"
# frames_to_animation(seq, output_path, fps=30, format="webp")

expr = part_1# gls.Intersection(mill_part_1, mill_part_2)
output = recursive_evaluate(expr.tensor(), sketcher, coords=coords)
output = output.cpu().numpy()
output = output.reshape(shape)[..., None]
fig = draw_solid_overlay(output, sketcher.resolution, figsize=figsize, color=part_color_1)
# fig.show()
img = fig_to_image(fig)
# img.save("renders/vis_1_odf_1.png")


expr = part_2
output = recursive_evaluate(expr.tensor(), sketcher, coords=coords)
output = output.cpu().numpy()
output = output.reshape(shape)[..., None]
fig = draw_solid_overlay(output, sketcher.resolution, figsize=figsize, color=part_color_2)
# fig.show()
img_1 = fig_to_image(fig)
together = overlay_image(img, img_1, position=(0, 0), mode="normal", opacity=1.0)
together.save("renders/vis_1_together_odf.png")



In [ ]:

# cur_translation = sample_points[i]
drill_bit = gls.Circle2D((DEFAULT_MILLBIT_SIZE/2.0,))

# scale, origin = get_expr_bounds(drill_bit.tensor(), sketcher)
scale = [0.5, 0.5]
scale = np.array(scale)
origin = np.array([0.0, 0.0])
scale = [x * 2 for x in scale]
coords, shape = sketcher.create_non_square_coords(scale, origin)
coords = coords.float()
figsize = (scale[1] * 3, scale[0] * 3)

output2 = recursive_evaluate(drill_bit.tensor(), sketcher, coords=coords)
output2 = output2.cpu().numpy()
output2 = output2.reshape(shape)[..., None]
fig = draw_solid_overlay(output2, sketcher.resolution, figsize=figsize, color=drillbit_color)
img2 = fig_to_image(fig)
img2.save("renders/vis_1_drill.png")
# output_path = "renders/vis_1_drill_1.webp"
# frames_to_animation(seq, output_path, fps=30, format="webp")